# Lesson 3: ⬇️ Download  

### 🎯 Learning Objectives
In this lesson, we will learn how to download the float time series and individual profiles identified using the filtered Argo index file (Lesson 1) and figures (Lesson 2).

### 🛠️ Prerequisites
Before starting this lesson, make sure that you have completed **Lesson 2**.
If you are using **Binder**, make sure to upload the relevant filtered index files created in **Lesson 1**.

### ❓ How to Use This Notebook
* 📚 **Read** the tutorial text blocks carefully, as they provide the essential background information behind the code.
* ▶️ **Run** each code cell sequentially by clicking the cell and pressing `Shift + Enter`.
* 📝 **Exercise** your knowledge! At the end of this notebook, we provide active learning exercises where you will need to write the code yourself.

### 🚀 Ready? Let's Get Started!
---

## 📚 Tutorials

### Import libraries

▶️ Run the cell below to import relevant Python libraries for this lesson.

In [ ]:
import pandas as pd
import os
import urllib.request

### Time series vs. individual profiles

Synthetic BGC-Argo data are provided as two types: time series and individual profiles. The former is simply a concatenation of the latter, therefore, the former is a 2D dataset (time, depth), whereas the former is a 1D dataset (depth). Below are examples of the two datasets:

* An example of time series: [6903574_Sprof.nc](https://data-argo.ifremer.fr/dac/coriolis/6903574/6903574_Sprof.nc)
* An example of individual profiles: [SD6903574_001.nc](https://data-argo.ifremer.fr/dac/coriolis/6903574/profiles/SD6903574_001.nc)

"Which datasets to use?" depends on your specific application. If you are interested in the the temporal changes, time series may be more suitable. In contrast, if you are interested in spatial distribution, individual profiles may be more suitable.

### The function `download_argo`

▶️ Run the cell below to define a function that downloads the selected float time series or profiles. To use this function, we need to provide the filtered index. If we want to download float time series (_Sprof.nc), we also need to provide the WMO(s) of the float(s). If we want to download the individual profiles (_XXX.nc), we do not need to provide the second argument (but note that it will download all profiles listed in the filtered index).

In [ ]:
def download_argo(df_index, wmoid_input=None):
    """
    This function downloads Argo data from one of the GDAC nodes. 
    
    INPUT:
    * df_index: the filtered index file
    * wmoid_input (optional): Either a string/integer or a list of strings/integers of WMO IDs.
                              If wmoid_input is provided, it will download Sprof time series.
                              If wmoid_input is not provided, it will download all profiles listed in df_index.
    
    OUTPUT:
    * netCDF files will be downloaded in ../floats/{wmo}/
    """
    
    # ==========================================
    # 1. Handle Optional Input & Warning
    # ==========================================
    if wmoid_input is None:
        num_files = len(df_index)
        print(f"⚠️ No specific floats provided.")
        print(f"⚠️ Defaulting to download ALL {num_files} files in the provided DataFrame!")
        
        # Extra safety net in case they passed the raw, unfiltered index by mistake!
        if num_files > 1000:
            print("🚨 WARNING: This is a massive download. Press the 'Stop' square in Jupyter if this was a mistake!")
            
        # Automatically grab the 'file' column from the DataFrame
        wmoid_input = df_index['file']

    # ==========================================
    # 2. Test and find a working server
    # ==========================================
    urls_to_try = [
        'https://data-argo.ifremer.fr/',
        'https://usgodae.org/pub/outgoing/argo/'
    ]
    
    active_url = None
    for base_url in urls_to_try:
        try:
            # Send a quick ping to see if the server is awake
            urllib.request.urlopen(base_url, timeout=5)
            active_url = base_url
            print(f"🌍 Successfully connected to {active_url}")
            break
        except Exception:
            print(f"❌ {base_url} is not responding...")

    if not active_url:
        print("🚨 Could not connect to any Argo servers. Please check your internet.")
        return

    # ==========================================
    # 3. Convert input to a standard Python list
    # ==========================================
    if isinstance(wmoid_input, pd.Series):
        # Handles Pandas columns smoothly
        item_list = wmoid_input.tolist()
        
    elif isinstance(wmoid_input, (int, str)):
        # If it's a single integer or string, safely wrap it in a list!
        item_list = [wmoid_input]
        
    else:
        # Assumes it is already a list, tuple, or array
        item_list = list(wmoid_input)

    # ==========================================
    # 4. Loop through the requested items
    # ==========================================
    for item in item_list:
        item_str = str(item)
        
        # ------------------------------------------
        # SCENARIO A: Individual Profile Download
        # Example: 'aoml/1901584/profiles/R1901584_001.nc'
        # ------------------------------------------
        if '.nc' in item_str or '/' in item_str:
            
            # Extract the WMO ID and filename from the long string
            parts = item_str.split('/')
            wmo_str = parts[1]       # e.g., '1901584'
            file_name = parts[-1]    # e.g., 'R1901584_001.nc'
            
            # Setup output directory -> "wmoid/profiles/"
            out_dir = os.path.join(wmo_str, 'profiles')
            os.makedirs(out_dir, exist_ok=True)
            
            # Construct URL and destination
            url = f"{active_url}dac/{item_str}"
            destination = os.path.join(out_dir, file_name)
            
        # ------------------------------------------
        # SCENARIO B: Time Series (Sprof) Download
        # Example: '1901584'
        # ------------------------------------------
        else:
            wmo_str = item_str
            
            # Look up the float in df_index to find its DAC (organization)
            match = df_index[df_index['file'].str.contains(f"/{wmo_str}/", na=False)]
            
            if match.empty:
                print(f"⚠️ WMO ID {wmo_str} not found in the index file. Skipping...")
                continue
                
            file_path = match['file'].values[0]
            orgname = file_path.split('/')[0] # e.g., 'aoml'
            
            # Setup output directory -> "../floats/wmoid/"
            out_dir = f'../floats/{wmo_str}'
            os.makedirs(out_dir, exist_ok=True)
            
            # Construct URL and destination
            file_name = f"{wmo_str}_Sprof.nc"
            url = f"{active_url}dac/{orgname}/{wmo_str}/{file_name}"
            destination = os.path.join(out_dir, file_name)

        # ------------------------------------------
        # 5. Perform the actual download
        # ------------------------------------------
        try:
            urllib.request.urlretrieve(url, destination)
            print(f"✅ Download successful: {destination}")
        except Exception as e:
            print(f"❌ Failed to download from {url}\n   Error: {e}")

### Load the filtered index file

▶️ Run the cell below to load the filtered index file created in Lesson 1.

In [ ]:
df_index = pd.read_parquet('../index_files/argo_synthetic-profile_index_default.parquet')
df_index

### Download the float time series

▶️ Run the cell below to download the time series for the float WMO **5905613**, which is one of the three floats available in this index file (see `../figures/map_default.png` created in Lesson 2).

In [ ]:
download_argo(df_index, 5906513)

### Download the individual profiles

▶️ Run the cell below to download the individual profiles included in the index file **after July 1, 2024** (during which the three floats have consistent profiling cycles; see `../figures/timeline_default.png` in Lesson 2).

In [ ]:
df_index_limit = df_index[df_index['datetime'] > pd.to_datetime("2024-07-01")]
download_argo(df_index_limit)

**This is the end of the tutorials for this lesson. Hope you enjoyed it!**

---

## 📝 Exercises

Using the filtered index files created in Lesson 1 and the figures created in Lesson 2, identify and download the float time series or individual profiles that meet specific criteria using the function `download_argo`.

### Exercise 1: Download the individual profiles of chlorophyll-a in the Southern Ocean on March 1, 2026

From `../figures/map_ex1.png` created in Lesson 2, we see the global coverage of chlorophyll-a measurements on March 1, 2026. Suppose we are only interested in the Southern Ocean. Refine the filtered index (`../index_files/argo_synthetic-profile_index_ex1.parquet` in Lesson 1) based on the latitude criterion (let's say that south of 30S is the Southern Ocean) and download the profiles.

📝 Write the code yourself below and ▶️ run it!

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python

df_index_ex1 = pd.read_parquet('argo_synthetic-profile_index_ex1.parquet')
df_index_so = df_index_ex1[df_index_ex1['latitude'] < -30]
download_argo(df_index_so, None)

### Exercise 2: Download the full-sensor float time series

Let's download the full-sensor time series of float **5905531** listed in `../index_files/argo_synthetic-profile_index_exercise2.parquet`.

📝 Write the code yourself below and ▶️ run it!

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python

df_index_ex2 = pd.read_parquet('../index_files/argo_synthetic-profile_index_ex2.parquet')
download_argo(df_index_ex2, '5905531')

### Exercise 3: Download the two sister floats that are slowly drifting near the Equator in the eastern Pacific

From Exercise 3 in Lesson 2, we found two floats (#20 and #21) that were operating near and at the same time and have the sequential WMO IDs. We are interested in how the float time series between these two floats compare. Let's refine the index (`../index_files/argo_synthetic-profile_index_ex3.parquet`) to download the two time series.

📝 Write the code yourself below and ▶️ run it!

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python

df_index_ex3 = pd.read_parquet('argo_synthetic-profile_index_ex3.parquet')
download_argo(df_index_ex3, [5906475, 5906476])


---

This is the end of the lesson. If you are using **Binder**, don't forget to dowload the files created in this lesson before you lose connection!

Well done 🎉, take a break 💤, have another cup ☕, and move to the next lesson ✍️ when you are ready 💪

While your memory is fresh, please feel free to provide your user experience on this lesson by visiting [this link](https://forms.gle/oAGmz5RTW4Pp46bt7). Thanks!